# 02 - Baseline training (YOLO11n)
Runtime > Change runtime type > **T4 GPU**. Run the cells **one at a time** (not Run all):
the Drive cell needs you to approve access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GH_TOKEN')
REPO = 'ThemistoklisSakkas/sar-ship-robustness'
DEST = '/content/sar-ship-robustness'
if os.path.isdir(DEST):
    !git -C {DEST} pull -q https://{token}@github.com/{REPO}.git main
else:
    !git clone -q https://{token}@github.com/{REPO}.git {DEST}
    !git -C {DEST} remote set-url origin https://github.com/{REPO}.git
%cd {DEST}
!pip install -q -r requirements.txt

DRIVE_ROOT = '/content/drive/MyDrive/sar-ship-robustness'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_ROOT = f'{DRIVE_ROOT}/runs'
NAME = 'baseline_yolo11n'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Train
About 30-60 min on a T4. Weights are saved on Drive after every epoch.
If the runtime disconnects, re-run the two cells above, then the **resume** cell.

In [ ]:
!python -m sarship.train --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME}

In [ ]:
# Only after a disconnect: continue from the last saved epoch
# !python -m sarship.train --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME} --resume

## Evaluate (val + held-out test, all / inshore / offshore) and draw examples

In [ ]:
!python -m sarship.evaluate --weights {RUNS_ROOT}/{NAME}/weights/best.pt --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --name {NAME}

In [ ]:
from IPython.display import Image, display
EVAL = f'{RUNS_ROOT}/{NAME}/evaluation'
for name in ['training_curves', 'examples_good', 'examples_bad']:
    print(name)
    display(Image(f'{EVAL}/{name}.png'))